# 05 — Cross-Event Weight Transfer

Implements [validation.md §5f](../docs/validation.md): the strongest single test of methodology generalization across events.

**Procedure** (per model):
1. Load the Russia 2022 fit (`02_Fit_Models` output) → get weights $w^{RU}$
2. Build the Hormuz 2026 panel with the same 21-donor shared pool
3. Apply $w^{RU}$ to the Hormuz donors: $\hat Y^{HZ}_t = \sum_j w^{RU}_j \cdot Y^{HZ}_{jt}$
4. Compare the transferred Hormuz counterfactual to the independently-fit Hormuz counterfactual

**Diagnostic interpretation:**
- Transferred gap ≈ independent gap → factor structure regime-stable 2020-22 → 2024-26 → strong evidence the methodology generalizes
- Transferred gap ≠ independent gap → regime drift; the independent Hormuz fit carries additional uncertainty beyond within-model variance

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from lib.config import DONOR_POOL_VARIANT
from lib.data import build_panel, load_fit, save_validation_table, load_gpr
from lib.plotting import plot_ensemble_paths, plot_ensemble_gaps, plot_paths

MODELS = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']
WINDOW = 'preferred'
VARIANT = DONOR_POOL_VARIANT   # must be 'shared' for weight transfer to work directly

if VARIANT != 'shared':
    print(f'WARNING: VARIANT = {VARIANT!r}, not "shared" — weight transfer needs the shared 21-donor pool.')

Last run: 2026-05-26 01:14:24


## Cross-event transfer per model

In [2]:
def project_with_weights(panel, treated, donors, weights, t0):
    """Apply pre-fit weights to a new panel to project a counterfactual."""
    full_mask = panel.index >= panel.index.min()  # all
    Y_full_donors = panel.loc[full_mask, donors].values
    w = weights.reindex(donors).fillna(0.0).values
    Y_hat = Y_full_donors @ w
    Y_actual = panel.loc[full_mask, treated].values
    return pd.Series(Y_actual - Y_hat, index=panel.index[full_mask], name='gap')

transfer_rows = []
transfer_fits = {}

for model in MODELS:
    russia_fit = load_fit('russia', WINDOW, model, variant=VARIANT)
    hormuz_fit = load_fit('hormuz', WINDOW, model, variant=VARIANT)
    if russia_fit is None or hormuz_fit is None:
        print(f'{model}: skipped (missing fits)')
        continue

    # Build the Hormuz panel
    hormuz_panel, hormuz_meta = build_panel('hormuz', WINDOW, variant=VARIANT)

    # Project Hormuz counterfactual using Russia's weights
    russia_weights = russia_fit['weights']
    transferred_gap = project_with_weights(hormuz_panel, 'Brent', hormuz_meta['donors'],
                                            russia_weights, t0=hormuz_meta['t0'])

    # Compute headline statistics for the transferred fit
    pre_mask = transferred_gap.index < hormuz_meta['t0']
    post_mask = transferred_gap.index >= hormuz_meta['t0']

    transferred_rmspe_pre = float(np.sqrt(np.mean(transferred_gap[pre_mask].values ** 2)))
    transferred_post_mean = float(100 * (np.exp(transferred_gap[post_mask].mean()) - 1))
    independent_post_mean = float(100 * (np.exp(hormuz_fit['gap'][hormuz_fit['gap'].index >= hormuz_meta['t0']].mean()) - 1))

    transfer_rows.append({
        'model': model,
        'russia_rmspe_pre': russia_fit['rmspe_pre'],
        'hormuz_independent_rmspe_pre': hormuz_fit['rmspe_pre'],
        'hormuz_transferred_rmspe_pre': transferred_rmspe_pre,
        'hormuz_independent_post_gap_pct': independent_post_mean,
        'hormuz_transferred_post_gap_pct': transferred_post_mean,
        'transferred_minus_independent_pct': transferred_post_mean - independent_post_mean,
    })

    # Build pseudo-fit dict for plotting comparison
    Y_hat_transferred = hormuz_panel.loc[transferred_gap.index, 'Brent'] - transferred_gap
    transferred_fit = {
        'model': f'{model}_transferred',
        'weights': russia_weights,
        'rmspe_pre': transferred_rmspe_pre,
        'actual': hormuz_fit['actual'],
        'synth': Y_hat_transferred,
        'gap': transferred_gap,
        't0': hormuz_meta['t0'],
        't_pre_start': hormuz_meta['t_pre_start'],
    }
    transfer_fits[model] = {'independent': hormuz_fit, 'transferred': transferred_fit}

transfer_df = pd.DataFrame(transfer_rows)
save_validation_table(transfer_df, 'cross_event_transfer')
transfer_df.round(3)

,model,russia_rmspe_pre,hormuz_independent_rmspe_pre,hormuz_transferred_rmspe_pre,hormuz_independent_post_gap_pct,hormuz_transferred_post_gap_pct,transferred_minus_independent_pct
0,convex_scm,0.106,0.066,0.109,38.554,46.960,8.406
1,ascm,0.106,0.066,0.144,43.666,27.268,-16.397
2,elastic_net,0.055,0.048,0.185,49.820,23.185,-26.635
3,xgboost,0.039,0.048,1.065,36.829,-58.171,-95.000
4,bayesian_ridge,0.043,0.035,13.707,43.607,-100.000,-143.607


Last run: 2026-05-26 01:14:24


## Visualize: transferred vs independent Hormuz counterfactual

Per model: solid lines are observed Brent; dashed lines are the two synthetic Brent counterfactuals (independent fit and Russia-transferred). Tight overlap = stable factor structure.

In [3]:
for model, pair in transfer_fits.items():
    fig = plot_ensemble_paths({'independent (Hormuz-fit)': pair['independent'],
                                'transferred (Russia weights)': pair['transferred']},
                              title=f'{model} — Hormuz counterfactuals: independent vs Russia-transferred')
    fig.show()

Last run: 2026-05-26 01:14:25


In [4]:
for model, pair in transfer_fits.items():
    fig = plot_ensemble_gaps({'independent (Hormuz-fit)': pair['independent'],
                               'transferred (Russia weights)': pair['transferred']},
                             title=f'{model} — Hormuz gap (%): independent vs Russia-transferred')
    fig.show()

Last run: 2026-05-26 01:14:25


## Interpretation

**Methodology generalizes if:** the `transferred_minus_independent_pct` column is small (e.g., within ±5%) for most models. This means Russia-era donor weights, applied to the Hormuz era, produce a synthetic close to the one Hormuz-era data fits independently. Factor structure is stable.

**Regime drift if:** the column is large or sign-inconsistent across models. The Hormuz estimate then has additional uncertainty beyond the within-model spread reported by `02_Fit_Models`.